## Finetune your own Speech-to-Text Whisper model on the language of your choice on a GPU, for free!

### Setup GPU
First, you'll need to enable GPUs for the notebook: Navigate to Edit→Notebook Settings Select T4 GPU from the Hardware Accelerator section Click Save and accept. Next, we'll confirm that we can connect to the GPU:

In [ ]:
import torch

if not torch.cuda.is_available():
    print("GPU NOT available!")
else:
    print("GPU is available!")

### Setup and login Hugging Face

The dataset we use for finetuning is Mozilla's [Common Voice](https://commonvoice.mozilla.org/).

In order to download the Common Voice dataset, track training and evaluation metrics of the finetuning and save your final model to use it and share it with others later, we will be using the Hugging Face (HF) platform. Before starting, make sure you:
1. have a HF [account](https://huggingface.co/join)
2. set up [personal access token](huggingface.co/settings/tokens)
3. login to hugging face in this notebook by running the command below and using your token


In [7]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 

In [5]:
from huggingface_hub import notebook_login

notebook_login()

### Download and install speech-to-text-finetune package

In [1]:
!git clone https://github.com/Mozilla-Data-Collective/speech-to-text-finetune

Cloning into 'speech-to-text-finetune'...
remote: Enumerating objects: 1524, done.
remote: Counting objects: 100% (598/598), done.
remote: Compressing objects: 100% (318/318), done.
remote: Total 1524 (delta 426), reused 322 (delta 270), pack-reused 926 (from 2)
Receiving objects: 100% (1524/1524), 6.94 MiB | 13.93 MiB/s, done.
Resolving deltas: 100% (748/748), done.


In [2]:
%cd speech-to-text-finetune/

/content/speech-to-text-finetune


In [4]:
!pip install --quiet -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for speech-to-text-finetune (pyproject.toml) ... done


In [6]:
!apt install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


***IMPORTANT:*** After installing the package, you need to restart the kernel / session: "Runtime -> Restart session" and then run the cells below

In [7]:
!cp example_data/.env.example src/speech_to_text_finetune/.env

In [8]:
from google.colab import userdata
import os

mdc_api_key = userdata.get('MDC_API_KEY')
env_file_path = 'src/speech_to_text_finetune/.env'

if mdc_api_key:
    # Read existing lines
    lines = []
    if os.path.exists(env_file_path):
        with open(env_file_path, 'r') as f:
            lines = f.readlines()

    found = False
    new_lines = []
    for line in lines:
        if line.startswith('MDC_API_KEY='):
            new_lines.append(f'MDC_API_KEY={mdc_api_key}\n')
            found = True
        else:
            new_lines.append(line)

    # If not found, append the new key
    if not found:
        new_lines.append(f'MDC_API_KEY={mdc_api_key}\n')

    # Write all lines back to the file
    with open(env_file_path, 'w') as f:
        f.writelines(new_lines)
    print("MDC_API_KEY successfully updated/written to .env file.")
else:
    print("MDC_API_KEY secret not found. Please add it to Colab secrets.")

MDC_API_KEY successfully updated/written to .env file.


In [10]:
!pip install python-dotenv

In [16]:
%cd speech-to-text-finetune/

/content/speech-to-text-finetune


In [26]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
# The .env file was previously saved at src/speech_to_text_finetune/.env
# Since we are currently in /content/speech-to-text-finetune/,
# we need to provide the relative path to the .env file.

print(f"Current working directory: {os.getcwd()}")
load_dotenv(dotenv_path='./src/speech_to_text_finetune/.env')

# Verify that the MDC_API_KEY is now set in the environment
if 'MDC_API_KEY' in os.environ:
    print("MDC_API_KEY successfully loaded into environment variables.")
else:
    print("MDC_API_KEY not found in environment variables after loading .env. Please check the .env file content.")

Current working directory: /content/speech-to-text-finetune
MDC_API_KEY successfully loaded into environment variables.


In [18]:
from speech_to_text_finetune.finetune_whisper import run_finetuning

**NOTE**: Certain "high-resource" languages like English or French have really big datasets (+50GB) which might fill up your disk storage fast. Make sure you have enough storage available before choosing a Common Voice language and finetuning on it.

In [39]:
# @title Finetuning configuration and hyperparameter setting
import yaml


def save_to_yaml(filename="config.yaml"):
    with open(filename, "w") as file:
        yaml.dump(cfg, file)


model_id = "openai/whisper-small"  # @param ["openai/whisper-tiny", "openai/whisper-small", "openai/whisper-medium","openai/whisper-large-v3"]
dataset_id = "cmn2e8rmi01l6mm07vxurptse"  # @param {type: "string"}
language = "Romanian"  # @param {type: "string"}
repo_name = "whisper-small-ro-finetuned"  # @param {type: "string"}
push_to_hub = True  # @param {type: 'boolean'}
n_train_samples = -1  # @param {type: "int"}
n_test_samples = -1  # @param {type: "int"}
hub_private_repo = True  # @param {type: 'boolean'}
max_steps = 50  # @param {type: "slider", min: 1, max: 3000, step: 10}
per_device_train_batch_size = 32  # @param {type: "slider", min: 1, max: 300}
gradient_accumulation_steps = 1  # @param {type: "slider", min: 1, max: 10}
warmup_steps = 50  # @param {type: "slider", min: 0, max: 500}
gradient_checkpointing = True  # @param {type: 'boolean'}
fp16 = True  # @param {type: 'boolean'}
per_device_eval_batch_size = 8  # @param {type: "slider", min: 1, max: 200}
save_steps = 5  # @param {type: "slider", min: 1, max: 500}
logging_steps = 5  # @param {type: "slider", min: 1, max: 500}
load_best_model_at_end = True  # @param {type: 'boolean'}

cfg = {
    "model_id": model_id,
    "dataset_id": dataset_id,
    "language": language,
    "repo_name": repo_name,
    "n_train_samples": n_train_samples,
    "n_test_samples": n_test_samples,
    "training_hp": {
        "push_to_hub": push_to_hub,
        "hub_private_repo": hub_private_repo,
        "max_steps": max_steps,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": 1e-5,
        "warmup_steps": warmup_steps,
        "gradient_checkpointing": gradient_checkpointing,
        "fp16": fp16,
        "eval_strategy": "steps",
        "per_device_eval_batch_size": per_device_eval_batch_size,
        "predict_with_generate": True,
        "generation_max_length": 225,
        "save_steps": save_steps,
        "logging_steps": logging_steps,
        "load_best_model_at_end": load_best_model_at_end,
        "save_total_limit": 1,
        "metric_for_best_model": "wer",
        "greater_is_better": False,
    },
}

save_to_yaml()

### Start finetuning job

Note that this might take a while, anything from 10min to 10hours depending on your model choice and hyper-parameter configuration

In [40]:
import os
from dotenv import load_dotenv

env_file_path = 'src/speech_to_text_finetune/.env'

# Read existing lines, remove MDC_API_URL if present
lines = []
if os.path.exists(env_file_path):
    with open(env_file_path, 'r') as f:
        lines = f.readlines()

new_lines = []
found_mdc_api_url = False
for line in lines:
    if line.strip().startswith('MDC_API_URL='):
        print("MDC_API_URL found and will be removed from .env file.")
        found_mdc_api_url = True
    else:
        new_lines.append(line)

if found_mdc_api_url:
    with open(env_file_path, 'w') as f:
        f.writelines(new_lines)
    print("MDC_API_URL successfully removed from .env file.")
else:
    print("MDC_API_URL not found in .env file (or already removed).")

# Reload environment variables after modifying .env file
# The current working directory is /content/speech-to-text-finetune
load_dotenv(dotenv_path='./src/speech_to_text_finetune/.env', override=True) # override=True ensures re-loading

# Verify that MDC_API_URL is no longer in os.environ, or if it was, it's using default logic.
# This check is just for confirmation, actual usage is by the downstream library
if 'MDC_API_URL' in os.environ:
    print(f"Warning: MDC_API_URL is still present in environment variables with value: {os.environ['MDC_API_URL']}")
else:
    print("MDC_API_URL is no longer explicitly set in environment variables (or was never).")

run_finetuning(config_path="config.yaml")

2026-04-23 17:15:05.424 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:65 - Finetuning starts soon, results saved locally at ./artifacts/whisper-small-ro-finetuned
2026-04-23 17:15:05.543 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:70 - Results will also be uploaded in HF at gandesc/whisper-small-ro-finetuned. Private repo is set to True.
2026-04-23 17:15:05.544 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:76 - Loading openai/whisper-small on CPU.


MDC_API_URL not found in .env file (or already removed).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

2026-04-23 17:15:06.988 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:82 - Configuring openai/whisper-small for Romanian
2026-04-23 17:15:08.580 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:120 - Loading cmiupa1t801hwmf07xcawi3ve.


█████████████████████████████████████████████████🦊 100.0% (66.3 MB/66.3 MB) Average: 22.0 MB/s Total time: 00:03


2026-04-23 17:15:14.347 | DEBUG    | speech_to_text_finetune.data_process:load_dataset_from_dataset_id:142 - MDC load skipped for cmiupa1t801hwmf07xcawi3ve: 
Unsupported split values found in dataset: 0000000001_0300000050_general, 3000000001_3000000300_chat, 4000000001_4000000300_customerservice


ValueError: There was an error trying to load the dataset: cmiupa1t801hwmf07xcawi3ve. If the dataset id is a valid MDC identifier, please check that:
- you have agreed to the terms & conditions of the specific dataset.
- you have set your MDC_API_KEY environment variable.
If the dataset id is a local path, please check that you are using the absolute path and it exists.